In [ ]:
!pip install -q openai


In [ ]:
# import libraries

import os
import re
import json
import json
import pandas as pd
from copy import deepcopy
from openai import OpenAI


In [ ]:
# mock input

mock_news = [
    {
        "title": "Nvidia reports strong quarterly earnings",
        "text": (
            "Nvidia reported quarterly revenue growth and earnings above "
            "analyst expectations. Data center demand remained strong. "
            "Management also warned that supply constraints could continue."
        ),
        "source": "Mock Financial News"
    },

    {
        "title": "Nvidia announces new AI partnerships",
        "text": (
            "Nvidia announced several new partnerships related to artificial "
            "intelligence infrastructure. The partnerships could expand demand "
            "for Nvidia GPUs and software."
        ),
        "source": "Mock Technology News"
    },

    {
        "title": "Semiconductor stocks fall as interest rates rise",
        "text": (
            "Semiconductor stocks declined as higher interest rates pressured "
            "technology shares. Nvidia shares also moved lower with the broader "
            "technology sector."
        ),
        "source": "Mock Market News"
    }
]


# **PART B TEST CODE**

In [ ]:
# prompt chaining

def preprocess_news(article):

    text = article["text"]

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [ ]:
def classify_news(text):

    text_lower = text.lower()

    earnings_keywords = [
        "earnings",
        "revenue",
        "quarterly",
        "guidance",
        "analyst expectations"
    ]

    market_keywords = [
        "interest rates",
        "market",
        "stocks",
        "shares",
        "sector",
        "technology shares"
    ]

    if any(word in text_lower for word in earnings_keywords):
        return "earnings"

    elif any(word in text_lower for word in market_keywords):
        return "market"

    else:
        return "general_news"


In [ ]:
def extract_key_information(text, category):

    sentences = re.split(r"(?<=[.!?])\s+", text)

    category_keywords = {

        "earnings": [
            "earnings",
            "revenue",
            "expectations",
            "guidance",
            "demand"
        ],

        "market": [
            "market",
            "shares",
            "stocks",
            "rates",
            "sector"
        ],

        "general_news": [
            "partnership",
            "artificial intelligence",
            "gpu",
            "software",
            "product"
        ]
    }

    keywords = category_keywords[category]

    selected = [
        sentence
        for sentence in sentences
        if any(keyword in sentence.lower() for keyword in keywords)
    ]

    # if nothing matches, just take the first two sentences
    if not selected:
        selected = sentences[:2]

    return " ".join(selected[:2])


In [ ]:
def get_sentiment(text):

    text_lower = text.lower()

    positive_words = [
        "strong",
        "growth",
        "above",
        "expand",
        "demand",
        "partnership"
    ]

    negative_words = [
        "declined",
        "lower",
        "pressure",
        "constraints",
        "warned",
        "fall"
    ]

    positive_score = sum(
        text_lower.count(word)
        for word in positive_words
    )

    negative_score = sum(
        text_lower.count(word)
        for word in negative_words
    )

    if positive_score > negative_score:
        return "positive"

    elif negative_score > positive_score:
        return "negative"

    else:
        return "neutral"


In [ ]:
def summarize_news(text, max_words=35):

    words = text.split()

    if len(words) <= max_words:
        return text

    return " ".join(words[:max_words]) + "..."


In [ ]:
def run_prompt_chain(article):

    cleaned_text = preprocess_news(article)

    category = classify_news(cleaned_text)

    extracted = extract_key_information(
        cleaned_text,
        category
    )

    sentiment = get_sentiment(cleaned_text)

    summary = summarize_news(extracted)

    return {
        "category": category,
        "summary": summary,
        "sentiment": sentiment,
        "source": article["source"],
        "title": article["title"]
    }


**Routing**

In [ ]:
def earnings_specialist(news_item):

    result = deepcopy(news_item)

    result["specialist"] = "Earnings Specialist"

    result["specialist_focus"] = (
        "Reviewed earnings results, revenue trends, "
        "expectations, and company guidance."
    )

    return result


In [ ]:
def news_specialist(news_item):

    result = deepcopy(news_item)

    result["specialist"] = "General News Specialist"

    result["specialist_focus"] = (
        "Reviewed company developments, products, "
        "partnerships, and strategic announcements."
    )

    return result


In [ ]:
def market_specialist(news_item):

    result = deepcopy(news_item)

    result["specialist"] = "Market Specialist"

    result["specialist_focus"] = (
        "Reviewed broader market conditions and factors "
        "that may influence the stock price."
    )

    return result


In [ ]:
def route_news(news_item):

    category = news_item["category"]

    if category == "earnings":
        return earnings_specialist(news_item)

    elif category == "market":
        return market_specialist(news_item)

    else:
        return news_specialist(news_item)


**Run Part B**

In [ ]:
def run_track_b(news_articles):

    results = []

    for article in news_articles:

        # Prompt chaining
        chained_result = run_prompt_chain(article)

        # Routing
        routed_result = route_news(chained_result)

        results.append(routed_result)

    return results


In [ ]:
track_b_output = run_track_b(mock_news)

print(json.dumps(track_b_output, indent=2))


[
  {
    "category": "earnings",
    "summary": "Nvidia reported quarterly revenue growth and earnings above analyst expectations. Data center demand remained strong.",
    "sentiment": "positive",
    "source": "Mock Financial News",
    "title": "Nvidia reports strong quarterly earnings",
    "specialist": "Earnings Specialist",
    "specialist_focus": "Reviewed earnings results, revenue trends, expectations, and company guidance."
  },
  {
    "category": "general_news",
    "summary": "Nvidia announced several new partnerships related to artificial intelligence infrastructure. The partnerships could expand demand for Nvidia GPUs and software.",
    "sentiment": "positive",
    "source": "Mock Technology News",
    "title": "Nvidia announces new AI partnerships",
    "specialist": "General News Specialist",
    "specialist_focus": "Reviewed company developments, products, partnerships, and strategic announcements."
  },
  {
    "category": "market",
    "summary": "Semiconductor st

# **PART A TEST CODE**

In [ ]:
def create_research_plan(symbol):

    return [
        f"Collect fundamentals for {symbol}",
        f"Calculate technical indicators for {symbol}",
        f"Review news and earnings information for {symbol}",
        "Combine quantitative and qualitative evidence",
        "Generate investment stance and confidence"
    ]


In [ ]:
research_plan = create_research_plan("NVDA")

for step in research_plan:
    print("-", step)


- Collect fundamentals for NVDA
- Calculate technical indicators for NVDA
- Review news and earnings information for NVDA
- Combine quantitative and qualitative evidence
- Generate investment stance and confidence


In [ ]:
def fundamentals_technical_tool(symbol):

    return {
        "symbol": symbol,

        "fundamentals": {
            "pe_ratio": 45.2,
            "revenue_growth_yoy": 0.42,
            "gross_margin": 0.73,
            "operating_margin": 0.58,
            "debt_to_equity": 0.18,
            "free_cash_flow_billions": 25.4
        },

        "technicals": {
            "current_price": 175.00,
            "ma_50": 182.00,
            "ma_200": 154.00,
            "pct_from_52_week_high": -0.11,
            "pct_from_52_week_low": 0.48,
            "recent_volatility": 0.34
        },

        "source": "Mock yfinance data"
    }


In [ ]:
market_data = fundamentals_technical_tool("NVDA")

print(json.dumps(market_data, indent=2))


{
  "symbol": "NVDA",
  "fundamentals": {
    "pe_ratio": 45.2,
    "revenue_growth_yoy": 0.42,
    "gross_margin": 0.73,
    "operating_margin": 0.58,
    "debt_to_equity": 0.18,
    "free_cash_flow_billions": 25.4
  },
  "technicals": {
    "current_price": 175.0,
    "ma_50": 182.0,
    "ma_200": 154.0,
    "pct_from_52_week_high": -0.11,
    "pct_from_52_week_low": 0.48,
    "recent_volatility": 0.34
  },
  "source": "Mock yfinance data"
}


In [ ]:
def generate_draft_analysis(
    symbol,
    market_data,
    news_results
):

    fundamentals = market_data["fundamentals"]
    technicals = market_data["technicals"]

    evidence = []
    risks = []

    score = 0

    if fundamentals["revenue_growth_yoy"] > 0.20:
        score += 1

        evidence.append(
            f"Revenue growth is approximately "
            f"{fundamentals['revenue_growth_yoy']:.0%} YoY."
        )

    if fundamentals["gross_margin"] > 0.60:
        score += 1

        evidence.append(
            f"Gross margin is approximately "
            f"{fundamentals['gross_margin']:.0%}."
        )

    if fundamentals["pe_ratio"] > 40:
        score -= 1

        risks.append(
            f"The P/E ratio of {fundamentals['pe_ratio']:.1f} "
            "suggests a relatively high valuation."
        )

    if technicals["current_price"] > technicals["ma_200"]:
        score += 1

        evidence.append(
            "The stock is trading above its 200-day moving average."
        )

    if technicals["current_price"] < technicals["ma_50"]:
        score -= 1

        risks.append(
            "The stock is currently trading below its "
            "50-day moving average."
        )

    positive_news = sum(
        item["sentiment"] == "positive"
        for item in news_results
    )

    negative_news = sum(
        item["sentiment"] == "negative"
        for item in news_results
    )

    if positive_news > negative_news:

        score += 1

        evidence.append(
            f"Recent news sentiment leaned positive "
            f"({positive_news} positive vs. "
            f"{negative_news} negative articles)."
        )

    elif negative_news > positive_news:

        score -= 1

        risks.append(
            "Recent news sentiment leaned negative."
        )

    if score >= 3:
        stance = "Bullish"

    elif score <= -2:
        stance = "Bearish"

    else:
        stance = "Neutral"

    if abs(score) >= 4:
        confidence = "High"

    elif abs(score) >= 2:
        confidence = "Medium"

    else:
        confidence = "Low"

    thesis = (
        f"{symbol} shows strong fundamental growth and several "
        f"positive long-term indicators, but valuation and "
        f"short-term technical weakness create meaningful risks."
    )

    sources = [
        market_data["source"]
    ]

    sources.extend(
        item["source"]
        for item in news_results
    )

    sources = list(dict.fromkeys(sources))

    return {
        "symbol": symbol,
        "stance": stance,
        "confidence": confidence,
        "thesis": thesis,
        "evidence_used": evidence,
        "key_risks": risks,
        "sources": sources
    }


In [ ]:
draft_analysis = generate_draft_analysis(
    symbol="NVDA",
    market_data=market_data,
    news_results=track_b_output
)

print(json.dumps(draft_analysis, indent=2))


{
  "symbol": "NVDA",
  "stance": "Neutral",
  "confidence": "Medium",
  "thesis": "NVDA shows strong fundamental growth and several positive long-term indicators, but valuation and short-term technical weakness create meaningful risks.",
  "evidence_used": [
    "Revenue growth is approximately 42% YoY.",
    "Gross margin is approximately 73%.",
    "The stock is trading above its 200-day moving average.",
    "Recent news sentiment leaned positive (2 positive vs. 1 negative articles)."
  ],
  "key_risks": [
    "The P/E ratio of 45.2 suggests a relatively high valuation.",
    "The stock is currently trading below its 50-day moving average."
  ],
  "sources": [
    "Mock yfinance data",
    "Mock Financial News",
    "Mock Technology News",
    "Mock Market News"
  ]
}


# **PART C**

**Function format_analysis turns json into plain text in case this is needed for Evaluator**

In [ ]:
def format_analysis(analysis):

    evidence_text = "\n".join(
        f"- {item}"
        for item in analysis["evidence_used"]
    )

    risks_text = "\n".join(
        f"- {item}"
        for item in analysis["key_risks"]
    )

    sources_text = "\n".join(
        f"- {item}"
        for item in analysis["sources"]
    )

    return f"""
Investment Analysis: {analysis['symbol']}

Stance: {analysis['stance']}
Confidence: {analysis['confidence']}

Thesis:
{analysis['thesis']}

Evidence:
{evidence_text}

Key Risks:
{risks_text}

Sources:
{sources_text}
""".strip()



draft_text = format_analysis(draft_analysis)

print(draft_text)


Investment Analysis: NVDA

Stance: Neutral
Confidence: Medium

Thesis:
NVDA shows strong fundamental growth and several positive long-term indicators, but valuation and short-term technical weakness create meaningful risks.

Evidence:
- Revenue growth is approximately 42% YoY.
- Gross margin is approximately 73%.
- The stock is trading above its 200-day moving average.
- Recent news sentiment leaned positive (2 positive vs. 1 negative articles).

Key Risks:
- The P/E ratio of 45.2 suggests a relatively high valuation.
- The stock is currently trading below its 50-day moving average.

Sources:
- Mock yfinance data
- Mock Financial News
- Mock Technology News
- Mock Market News


**Choose either json or plain text for input into Evaluator**

In [ ]:
# json
analysis_to_evaluate = draft_analysis

# plain text
#analysis_to_evaluate = draft_text


**Create weak draft to test for catching problems**

In [ ]:
weak_draft = {
    "symbol": "NVDA",

    "stance": "Bullish",

    "confidence": "High",

    "thesis": (
        "Nvidia looks like a good investment because "
        "the company is doing well."
    ),

    "evidence_used": [
        "Revenue is growing."
    ],

    "key_risks": [],

    "sources": []
}

weak_draft_text = format_analysis(weak_draft)

print(weak_draft_text)


Investment Analysis: NVDA

Stance: Bullish
Confidence: High

Thesis:
Nvidia looks like a good investment because the company is doing well.

Evidence:
- Revenue is growing.

Key Risks:


Sources:


**Connect to OpenRouter**

In [ ]:
# Load OpenRouter API key

try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
except:
    OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")


if not OPENROUTER_API_KEY:
    raise ValueError(
        "OpenRouter API key not found. "
        "Add OPENROUTER_API_KEY to Colab Secrets."
    )


client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)


# model can easily be changed later
MODEL = "openai/gpt-4o-mini"
# MODEL = "openrouter/free"

## NOTE: WHEN USING OPENROUTER/FREE, SEARCH FOR
#3 response_format=OPTIMIZER_RESPONSE_FORMAT AND
## response_format=EVALUATION_RESPONSE_FORMAT, THEN
## ENSURE THIS IS BEING USED INSTEAD OF response_format=JSON_RESPONSE_FORMAT


**Generic LLM Function**

In [ ]:
def call_llm(
    system_prompt,
    user_prompt,
    temperature=0.2,
    response_format=None,
    max_attempts=3
):
    """
    Sends a prompt through OpenRouter.

    Automatically retries if a free model returns
    an empty or missing response.
    """

    last_model = None

    for attempt in range(1, max_attempts + 1):

        kwargs = {
            "model": MODEL,

            "messages": [
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": user_prompt
                }
            ],

            "temperature": temperature,

            # give the model enough room to return JSON
            "max_tokens": 1200,

            "extra_body": {
                "provider": {
                    "require_parameters": True
                }
            }
        }

        if response_format is not None:
            kwargs["response_format"] = response_format

        response = client.chat.completions.create(
            **kwargs
        )

        # store model name for debugging
        last_model = getattr(
            response,
            "model",
            "unknown"
        )

        message = response.choices[0].message
        content = message.content

        # successful response
        if content is not None and content.strip():

            if attempt > 1:
                print(
                    f"Successful response on "
                    f"attempt {attempt}."
                )

            return content

        # empty response
        print(
            f"Attempt {attempt}: "
            f"{last_model} returned no text. "
            "Retrying..."
        )

    # if all attempts fail
    raise ValueError(
        "OpenRouter returned no usable text after "
        f"{max_attempts} attempts. "
        f"Last model used: {last_model}"
    )


**JSON Parser**

In [ ]:
def parse_json_response(response_text):
    """
    Converts an LLM response into a Python dictionary.
    """

    if response_text is None:
        raise ValueError(
            "The model returned no text response."
        )

    text = response_text.strip()

    if not text:
        raise ValueError(
            "The model returned an empty response."
        )

    # remove markdown code fences if present
    text = re.sub(
        r"^```json\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"^```\s*",
        "",
        text
    )

    text = re.sub(
        r"\s*```$",
        "",
        text
    )

    try:
        return json.loads(text)

    except json.JSONDecodeError:

        # fall back to searching for JSON
        start = text.find("{")
        end = text.rfind("}")

        if start == -1 or end == -1:

            print("RAW MODEL RESPONSE:")
            print(repr(response_text))

            raise ValueError(
                "No JSON object found in model response."
            )

        json_text = text[start:end + 1]

        try:
            return json.loads(json_text)

        except json.JSONDecodeError as error:

            print("RAW MODEL RESPONSE:")
            print(response_text)

            raise ValueError(
                f"Model returned invalid JSON: {error}"
            )


**Validate Part A Input**

In [ ]:
REQUIRED_ANALYSIS_FIELDS = [
    "symbol",
    "stance",
    "confidence",
    "thesis",
    "evidence_used",
    "key_risks",
    "sources"
]


def validate_analysis(analysis):
    """
    Makes sure the Agent Core supplied all fields
    required by the evaluator-optimizer.
    """

    missing = [
        field
        for field in REQUIRED_ANALYSIS_FIELDS
        if field not in analysis
    ]

    if missing:
        raise ValueError(
            f"Analysis is missing required fields: {missing}"
        )

    return True


In [ ]:
# test, will yield true if correct
validate_analysis(draft_analysis)


True

**Evaluator**

In [ ]:
JSON_RESPONSE_FORMAT = {
    "type": "json_object"
}


In [ ]:
EVALUATION_RESPONSE_FORMAT = {
    "type": "json_schema",

    "json_schema": {
        "name": "investment_analysis_evaluation",

        "strict": True,

        "schema": {
            "type": "object",

            "properties": {

                "scores": {
                    "type": "object",

                    "properties": {
                        "thesis_quality": {
                            "type": "integer"
                        },

                        "evidence_support": {
                            "type": "integer"
                        },

                        "risk_analysis": {
                            "type": "integer"
                        },

                        "consistency": {
                            "type": "integer"
                        },

                        "source_quality": {
                            "type": "integer"
                        },

                        "actionability": {
                            "type": "integer"
                        }
                    },

                    "required": [
                        "thesis_quality",
                        "evidence_support",
                        "risk_analysis",
                        "consistency",
                        "source_quality",
                        "actionability"
                    ],

                    "additionalProperties": False
                },

                "strengths": {
                    "type": "array",
                    "items": {
                        "type": "string"
                    }
                },

                "issues": {
                    "type": "array",
                    "items": {
                        "type": "string"
                    }
                },

                "recommendations": {
                    "type": "array",
                    "items": {
                        "type": "string"
                    }
                },

                "critical_issues": {
                    "type": "array",
                    "items": {
                        "type": "string"
                    }
                }
            },

            "required": [
                "scores",
                "strengths",
                "issues",
                "recommendations",
                "critical_issues"
            ],

            "additionalProperties": False
        }
    }
}


In [ ]:
def evaluate_analysis(
    analysis,
    pass_score=8.0,
    max_attempts=3
):
    """
    Evaluates the quality of an investment analysis.

    Automatically retries if the LLM returns incomplete
    evaluator JSON.
    """

    validate_analysis(analysis)

    system_prompt = """
You are the Evaluator Agent in a multi-agent financial analysis system.

Your job is to critically evaluate a draft investment analysis.

Evaluate only the information contained in the draft.
Do NOT introduce new financial facts.
Do NOT perform outside research.
Do NOT assume missing evidence exists.

Score EACH of the following six categories from 1 to 5.

YOU MUST RETURN ALL SIX SCORES:

1. thesis_quality
2. evidence_support
3. risk_analysis
4. consistency
5. source_quality
6. actionability

Score meanings:

1 = very weak
2 = weak
3 = acceptable
4 = strong
5 = excellent

Definitions:

thesis_quality:
Is the investment thesis clear, specific, and meaningful?

evidence_support:
Is the stance supported by specific evidence contained
in the analysis?

risk_analysis:
Are meaningful risks clearly identified and discussed?

consistency:
Are the stance, confidence, thesis, evidence, and risks
internally consistent?

source_quality:
Does the analysis identify sources for the evidence it uses?
Do not independently verify those sources.

actionability:
Does the analysis provide a clear Bullish, Neutral, or Bearish
stance with an appropriate confidence level?

Return ONLY valid JSON in this exact structure:

{
    "scores": {
        "thesis_quality": 1,
        "evidence_support": 1,
        "risk_analysis": 1,
        "consistency": 1,
        "source_quality": 1,
        "actionability": 1
    },
    "strengths": [],
    "issues": [],
    "recommendations": [],
    "critical_issues": []
}

All six score fields are REQUIRED.

A critical issue should only be used for major problems such as:

- stance contradicts the evidence
- confidence is unsupported
- important required fields are effectively missing
- claims appear unsupported by the supplied evidence
- no meaningful risk discussion exists

Be specific and constructive.
"""

    base_user_prompt = f"""
Evaluate the following investment analysis:

{json.dumps(analysis, indent=2)}
"""

    last_error = None

    # retry incomplete evaluator responses
    for attempt in range(1, max_attempts + 1):

        user_prompt = base_user_prompt

        if attempt > 1:
            user_prompt += """

IMPORTANT:
Your previous response did not follow the required structure.

Make sure the new response contains ALL SIX score fields:

- thesis_quality
- evidence_support
- risk_analysis
- consistency
- source_quality
- actionability

Do not omit any required field.
"""

        response = call_llm(
            system_prompt,
            user_prompt,
            temperature=0.1,
            response_format=JSON_RESPONSE_FORMAT
        )

        try:
            evaluation = parse_json_response(
                response
            )

            validate_evaluation(
                evaluation
            )

            # valid response — leave retry loop
            break

        except ValueError as error:

            last_error = error

            print(
                f"Evaluator attempt {attempt} "
                f"returned incomplete output: {error}"
            )

            if attempt == max_attempts:
                raise ValueError(
                    "Evaluator failed to return a complete "
                    f"response after {max_attempts} attempts. "
                    f"Last error: {last_error}"
                )

    # calculate overall score
    scores = evaluation["scores"]

    overall_score = (
        sum(scores.values())
        / len(scores)
    ) * 2

    overall_score = round(
        overall_score,
        2
    )

    # determine pass / fail
    critical_issues = evaluation.get(
        "critical_issues",
        []
    )

    passed = (
        overall_score >= pass_score
        and len(critical_issues) == 0
    )

    evaluation["overall_score"] = overall_score
    evaluation["passed"] = passed

    return evaluation


In [ ]:
def validate_evaluation(evaluation):
    """
    Checks that the evaluator returned the structure
    required by the Evaluator-Optimizer workflow.
    """

    required_fields = [
        "scores",
        "strengths",
        "issues",
        "recommendations",
        "critical_issues"
    ]

    missing = [
        field
        for field in required_fields
        if field not in evaluation
    ]

    if missing:
        raise ValueError(
            f"Evaluation missing required fields: {missing}"
        )


    required_scores = [
        "thesis_quality",
        "evidence_support",
        "risk_analysis",
        "consistency",
        "source_quality",
        "actionability"
    ]

    missing_scores = [
        score
        for score in required_scores
        if score not in evaluation["scores"]
    ]

    if missing_scores:
        raise ValueError(
            f"Evaluation missing scores: {missing_scores}"
        )


    # make sure scores are valid
    for name, score in evaluation["scores"].items():

        if not isinstance(score, (int, float)):
            raise ValueError(
                f"{name} score must be numeric."
            )

        if score < 1 or score > 5:
            raise ValueError(
                f"{name} score must be between 1 and 5."
            )

    return True


In [ ]:
# test Evaluator on weak draft
weak_evaluation = evaluate_analysis(weak_draft)

print(
    json.dumps(
        weak_evaluation,
        indent=2
    )
)


{
  "scores": {
    "thesis_quality": 2,
    "evidence_support": 2,
    "risk_analysis": 1,
    "consistency": 2,
    "source_quality": 1,
    "actionability": 3
  },
  "strengths": [],
  "issues": [
    "The thesis is vague and lacks specific details about what makes Nvidia a good investment.",
    "The evidence provided is minimal and does not substantiate the bullish stance effectively.",
    "There is no discussion of meaningful risks associated with the investment.",
    "No sources are cited to support the claims made in the analysis."
  ],
  "recommendations": [
    "Provide a more detailed and specific investment thesis.",
    "Include more substantial evidence to support the bullish stance, such as financial metrics or market trends.",
    "Identify and discuss potential risks that could impact the investment.",
    "Cite credible sources to enhance the reliability of the analysis."
  ],
  "critical_issues": [
    "The lack of meaningful risk discussion is a major problem, as 

**Optimizer**

In [ ]:
OPTIMIZER_RESPONSE_FORMAT = {
    "type": "json_schema",

    "json_schema": {
        "name": "optimized_investment_analysis",

        "strict": True,

        "schema": {
            "type": "object",

            "properties": {

                "symbol": {
                    "type": "string"
                },

                "stance": {
                    "type": "string",
                    "enum": [
                        "Bullish",
                        "Neutral",
                        "Bearish"
                    ]
                },

                "confidence": {
                    "type": "string",
                    "enum": [
                        "Low",
                        "Medium",
                        "High"
                    ]
                },

                "thesis": {
                    "type": "string"
                },

                "evidence_used": {
                    "type": "array",
                    "items": {
                        "type": "string"
                    }
                },

                "key_risks": {
                    "type": "array",
                    "items": {
                        "type": "string"
                    }
                },

                "sources": {
                    "type": "array",
                    "items": {
                        "type": "string"
                    }
                }
            },

            "required": [
                "symbol",
                "stance",
                "confidence",
                "thesis",
                "evidence_used",
                "key_risks",
                "sources"
            ],

            "additionalProperties": False
        }
    }
}


In [ ]:
def optimize_analysis(analysis, evaluation):
    """
    Revises an investment analysis using feedback
    from the Evaluator Agent.
    """

    validate_analysis(analysis)

    original_symbol = analysis["symbol"]

    system_prompt = """
You are the Optimizer Agent in a multi-agent financial analysis system.

Your job is to improve an investment analysis using feedback
from an Evaluator Agent.

IMPORTANT RULES:

1. Use ONLY facts, evidence, risks, and sources already contained
   in the original analysis.
2. Do NOT invent financial numbers.
3. Do NOT invent news events.
4. Do NOT invent sources.
5. Do NOT perform outside research.
6. You may change stance or confidence if the existing evidence
   does not support the original conclusion.
7. Improve clarity, reasoning, risk discussion, and consistency.
8. Keep the same stock symbol.
9. If evidence is weak, reduce confidence rather than inventing
   additional evidence.

Return the revised analysis in the required structured format.
"""

    user_prompt = f"""
ORIGINAL ANALYSIS:

{json.dumps(analysis, indent=2)}


EVALUATOR FEEDBACK:

{json.dumps(evaluation, indent=2)}


Revise the analysis based on the evaluator feedback.
"""

    response = call_llm(
        system_prompt,
        user_prompt,
        temperature=0.2,
        #response_format=OPTIMIZER_RESPONSE_FORMAT
        response_format=JSON_RESPONSE_FORMAT
    )

    revised_analysis = parse_json_response(
        response
    )

    # normalize common model wording variations
    revised_analysis = normalize_analysis_labels(
        revised_analysis
    )

    validate_analysis(
        revised_analysis
    )

    # make sure optimizer did not change companies
    if revised_analysis["symbol"] != original_symbol:
        raise ValueError(
            "Optimizer changed the stock symbol from "
            f"{original_symbol} to "
            f"{revised_analysis['symbol']}."
        )

    return revised_analysis


In [ ]:
def normalize_analysis_labels(analysis):
    """
    Normalizes common LLM wording variations so they match
    the required interface contract.
    """

    analysis = deepcopy(analysis)

    # normalize stance
    stance = str(
        analysis.get("stance", "")
    ).strip().lower()

    # bullish labels
    if (
        "bull" in stance
        or "buy" in stance
        or "outperform" in stance
        or "accumulate" in stance
    ):
        analysis["stance"] = "Bullish"

    # bearish labels
    elif (
        "bear" in stance
        or "sell" in stance
        or "underperform" in stance
        or "reduce" in stance
    ):
        analysis["stance"] = "Bearish"

    # neutral labels
    elif (
        "neutral" in stance
        or "hold" in stance
        or "market perform" in stance
        or "mixed" in stance
    ):
        analysis["stance"] = "Neutral"

    # normalize confidence
    confidence = str(
        analysis.get("confidence", "")
    ).strip().lower()

    if "low" in confidence:
        analysis["confidence"] = "Low"

    elif (
        "medium" in confidence
        or "moderate" in confidence
    ):
        analysis["confidence"] = "Medium"

    elif "high" in confidence:
        analysis["confidence"] = "High"

    return analysis


In [ ]:
def validate_analysis(analysis):

    required_fields = [
        "symbol",
        "stance",
        "confidence",
        "thesis",
        "evidence_used",
        "key_risks",
        "sources"
    ]

    missing = [
        field
        for field in required_fields
        if field not in analysis
    ]

    if missing:
        raise ValueError(
            f"Analysis is missing required fields: {missing}"
        )


    valid_stances = [
        "Bullish",
        "Neutral",
        "Bearish"
    ]

    valid_confidence = [
        "Low",
        "Medium",
        "High"
    ]


    if analysis["stance"] not in valid_stances:
        raise ValueError(
            f"Invalid stance: {analysis['stance']}"
        )


    if analysis["confidence"] not in valid_confidence:
        raise ValueError(
            f"Invalid confidence: {analysis['confidence']}"
        )


    for field in [
        "evidence_used",
        "key_risks",
        "sources"
    ]:

        if not isinstance(analysis[field], list):
            raise ValueError(
                f"{field} must be a list."
            )


    return True


In [ ]:
# test Optimizer on weak draft
weak_evaluation = evaluate_analysis(
    weak_draft
)

improved_draft = optimize_analysis(
    weak_draft,
    weak_evaluation
)

print(
    json.dumps(
        improved_draft,
        indent=2
    )
)


BadRequestError: Error code: 400 - {'error': {'message': 'Provider returned error', 'code': 400, 'metadata': {'raw': '{\n  "error": {\n    "message": "\'messages\' must contain the word \'json\' in some form, to use \'response_format\' of type \'json_object\'.",\n    "type": "invalid_request_error",\n    "param": "messages",\n    "code": null\n  }\n}', 'provider_name': 'Azure', 'is_byok': False, 'previous_errors': [{'code': 400, 'message': 'Provider returned error', 'provider_name': 'OpenAI', 'raw': '{\n  "error": {\n    "message": "\'messages\' must contain the word \'json\' in some form, to use \'response_format\' of type \'json_object\'.",\n    "type": "invalid_request_error",\n    "param": "messages",\n    "code": null\n  }\n}'}]}}, 'user_id': 'user_3KIpTINaYgGWyLRx38831Bv6KxB'}

**Evaluator-Optimizer Loop**

In [ ]:
def run_evaluator_optimizer(
    initial_analysis,
    max_refinements=2,
    pass_score=8.0,
    verbose=True
):
    """
    Runs the complete Evaluator-Optimizer workflow.
    Maximum number of optimization/refinement
    rounds is controlled by max_refinements.
    """

    current_analysis = normalize_analysis_labels(
        deepcopy(initial_analysis)
    )

    validate_analysis(current_analysis)

    history = []

    # initial evaluation
    evaluation = evaluate_analysis(
        current_analysis,
        pass_score=pass_score
    )

    history.append({
        "stage": "Initial Draft",
        "analysis": deepcopy(current_analysis),
        "evaluation": deepcopy(evaluation)
    })

    if verbose:

        print("=" * 60)
        print("INITIAL EVALUATION")
        print("=" * 60)

        print(
            f"Score: "
            f"{evaluation['overall_score']}/10"
        )

        print(
            f"Passed: "
            f"{evaluation['passed']}"
        )

    # refinement loop
    for refinement_round in range(
        1,
        max_refinements + 1
    ):

        # stop early if analysis already passes
        if evaluation["passed"]:

            if verbose:
                print(
                    "\nQuality threshold met. "
                    "No additional refinement needed."
                )

            break

        if verbose:

            print("\n" + "=" * 60)

            print(
                f"REFINEMENT ROUND "
                f"{refinement_round}"
            )

            print("=" * 60)

            print("\nEvaluator recommendations:")

            for recommendation in evaluation.get(
                "recommendations",
                []
            ):
                print(
                    f"- {recommendation}"
                )

        # optimize analysis
        current_analysis = optimize_analysis(
            current_analysis,
            evaluation
        )

        # re-evaluate revised analysis
        evaluation = evaluate_analysis(
            current_analysis,
            pass_score=pass_score
        )

        history.append({
            "stage": f"Refinement {refinement_round}",
            "analysis": deepcopy(current_analysis),
            "evaluation": deepcopy(evaluation)
        })

        if verbose:

            print(
                f"\nNew Score: "
                f"{evaluation['overall_score']}/10"
            )

            print(
                f"Passed: "
                f"{evaluation['passed']}"
            )

    return {
        "final_analysis": current_analysis,
        "final_evaluation": evaluation,
        "history": history,
        "refinements_used": len(history) - 1
    }


In [ ]:
# test Loop on weak draft
weak_result = run_evaluator_optimizer(
    weak_draft,
    max_refinements=2,
    pass_score=8.0
)

print("\nFINAL ANALYSIS")

print(
    json.dumps(
        weak_result["final_analysis"],
        indent=2
    )
)

print("\nFINAL EVALUATION")

print(
    json.dumps(
        weak_result["final_evaluation"],
        indent=2
    )
)


**Run Evaluator-Optimizer Loop on Parts A/B input**

In [ ]:
result = run_evaluator_optimizer(
    draft_analysis,
    max_refinements=2,
    pass_score=8.0
)

final_analysis = result[
    "final_analysis"
]

print(
    json.dumps(
        final_analysis,
        indent=2
    )
)


**Evaluate the Evaluator-Optimizer Loop**

In [ ]:
def create_evaluation_table(result):

    rows = []

    for item in result["history"]:

        evaluation = item["evaluation"]
        scores = evaluation["scores"]

        rows.append({

            "Stage": item["stage"],

            "Thesis": scores[
                "thesis_quality"
            ],

            "Evidence": scores[
                "evidence_support"
            ],

            "Risk Analysis": scores[
                "risk_analysis"
            ],

            "Consistency": scores[
                "consistency"
            ],

            "Sources": scores[
                "source_quality"
            ],

            "Actionability": scores[
                "actionability"
            ],

            "Overall Score": evaluation[
                "overall_score"
            ],

            "Passed": evaluation[
                "passed"
            ]
        })

    return pd.DataFrame(rows)


In [ ]:
evaluation_df = create_evaluation_table(
    result
)

display(evaluation_df)
